### 문제 7-1 : LangGraph ReAct Agent 실습 연습문제 (Vector DB + Tool 연동)

In [ ]:
from typing import Literal
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
import json
import os

llm = ChatGroq(
    model="llama3-70b-8192",  
    temperature=0.1,
    max_tokens=1024
)

In [28]:
@tool
def search_cafe_menu(query: str) -> str:
    """
    카페 메뉴를 검색하는 도구입니다.
    
    Args:
        query: 검색할 메뉴나 카테고리 (예: '아메리카노', '라떼', '디저트', '티라미수' 등)
    
    Returns:
        검색 결과를 JSON 형식으로 반환
    """
    query_lower = query.lower()
    
    # 아메리카노 관련 검색
    if '아메리카노' in query_lower:
        return json.dumps({
            "category": "coffee",
            "results": [
                {
                    "name": "아메리카노",
                    "price": 4500,
                    "description": "진한 에스프레소에 뜨거운 물을 넣어 만든 클래식 커피",
                    "temperature": "HOT",
                    "ingredients": ["에스프레소", "뜨거운 물"],
                    "caffeine": "높음"
                },
                {
                    "name": "아이스 아메리카노",
                    "price": 5000,
                    "description": "진한 에스프레소에 차가운 물과 얼음을 넣어 시원하게 즐기는 커피",
                    "temperature": "COLD",
                    "ingredients": ["에스프레소", "차가운 물", "얼음"],
                    "caffeine": "높음"
                }
            ]
        }, ensure_ascii=False, indent=2)
    
    # 라떼 관련 검색
    elif '라떼' in query_lower:
        return json.dumps({
            "category": "latte",
            "results": [
                {
                    "name": "카페라떼",
                    "price": 5500,
                    "description": "부드러운 스팀밀크와 에스프레소의 완벽한 조화",
                    "temperature": "HOT",
                    "ingredients": ["에스프레소", "스팀밀크"],
                    "caffeine": "중간"
                },
                {
                    "name": "아이스 카페라떼",
                    "price": 6000,
                    "description": "시원한 우유와 에스프레소로 만든 부드러운 아이스 라떼",
                    "temperature": "COLD",
                    "ingredients": ["에스프레소", "차가운 우유", "얼음"],
                    "caffeine": "중간"
                },
                {
                    "name": "바닐라 라떼",
                    "price": 6500,
                    "description": "달콤한 바닐라 시럽이 들어간 프리미엄 라떼",
                    "temperature": "HOT/COLD",
                    "ingredients": ["에스프레소", "스팀밀크", "바닐라 시럽"],
                    "caffeine": "중간"
                },
                {
                    "name": "녹차 라떼",
                    "price": 6000,
                    "description": "진한 녹차와 부드러운 우유의 조화로운 맛",
                    "temperature": "HOT/COLD",
                    "ingredients": ["녹차 파우더", "스팀밀크"],
                    "caffeine": "낮음"
                }
            ]
        }, ensure_ascii=False, indent=2)
    
    # 디저트/티라미수 관련 검색
    elif '디저트' in query_lower or '티라미수' in query_lower:
        return json.dumps({
            "category": "dessert",
            "results": [
                {
                    "name": "티라미수",
                    "price": 7500,
                    "description": "이탈리아 전통 디저트로 마스카포네 치즈와 커피의 완벽한 조화",
                    "ingredients": ["마스카포네 치즈", "레이디핑거", "에스프레소", "코코아 파우더"],
                    "allergens": ["유제품", "글루텐"],
                    "serving_size": "1인분",
                    "special_notes": "커피 맛이 강하고 부드러운 질감이 특징"
                },
                {
                    "name": "치즈케이크",
                    "price": 6500,
                    "description": "진한 크림치즈로 만든 부드러운 케이크",
                    "ingredients": ["크림치즈", "설탕", "계란", "비스킷"],
                    "allergens": ["유제품", "글루텐", "계란"]
                }
            ]
        }, ensure_ascii=False, indent=2)
    
    # 가격대별 검색
    elif '가격' in query_lower or '저렴' in query_lower or '비싼' in query_lower:
        return json.dumps({
            "category": "price_range",
            "results": {
                "budget_friendly": [
                    {"name": "아메리카노", "price": 4500},
                    {"name": "아이스티", "price": 4000},
                    {"name": "레몬에이드", "price": 4500}
                ],
                "mid_range": [
                    {"name": "아이스 아메리카노", "price": 5000},
                    {"name": "카페라떼", "price": 5500},
                    {"name": "녹차 라떼", "price": 6000}
                ],
                "premium": [
                    {"name": "바닐라 라떼", "price": 6500},
                    {"name": "치즈케이크", "price": 6500},
                    {"name": "티라미수", "price": 7500}
                ]
            }
        }, ensure_ascii=False, indent=2)
    
    else:
        return json.dumps({
            "message": "검색 결과를 찾을 수 없습니다. '아메리카노', '라떼', '디저트', '티라미수' 등으로 검색해보세요."
        }, ensure_ascii=False)

In [29]:
class AgentState(MessagesState):
    """
    카페 Agent의 상태를 정의합니다.
    MessagesState를 상속하여 메시지 히스토리를 자동으로 관리합니다.
    """
    pass


In [ ]:
def cafe_agent_node(state: AgentState):
    """
    카페 메뉴 관련 질문에 답변하는 GROQ LLM Agent 노드
    """
    # 시스템 메시지 정의
    system_message = SystemMessage(content="""
    당신은 카페 메뉴 전문 상담사입니다.
    
    **역할과 책임:**
    - 카페 메뉴에 대한 상세한 정보 제공
    - 고객의 취향에 맞는 메뉴 추천
    - 가격, 재료, 제조 방법 등 구체적인 정보 안내
    
    **응답 가이드라인:**
    1. 메뉴 정보가 필요한 경우 search_cafe_menu 도구를 반드시 사용하세요
    2. 친근하고 전문적인 톤으로 답변하세요
    3. 가격은 원화(₩)로 표시하세요
    4. 메뉴의 특징과 차이점을 명확히 설명하세요
    5. 도구 호출 없이 추측하지 마세요
    
    **주요 메뉴 카테고리:**
    - 커피류: 아메리카노, 라떼 계열
    - 디저트: 티라미수, 치즈케이크 등
    - 가격대: ₩4,000 ~ ₩7,500
    
    **도구 사용 규칙:**
    - 아메리카노에 대한 질문: search_cafe_menu("아메리카노")
    - 라떼에 대한 질문: search_cafe_menu("라떼")
    - 디저트에 대한 질문: search_cafe_menu("디저트")
    - 가격에 대한 질문: search_cafe_menu("가격")
    """)
    
    llm_with_tools = llm.bind_tools([search_cafe_menu])
    
    messages = [system_message] + state["messages"]
    
    response = llm_with_tools.invoke(messages)
    
    return {"messages": [response]}


In [ ]:
def build_cafe_react_agent():
    """
    사용자 정의 ReAct Agent를 구성합니다.
    """
    # StateGraph 빌더 생성
    builder = StateGraph(AgentState)
    
    # 노드 추가
    builder.add_node("agent", cafe_agent_node)
    builder.add_node("tools", ToolNode([search_cafe_menu]))
    
    # 시작점 설정
    builder.set_entry_point("agent")
    
    builder.add_conditional_edges(
        "agent",
        tools_condition, 
    )
    
    builder.add_edge("tools", "agent")
    
    memory = MemorySaver()
    
    graph = builder.compile(checkpointer=memory)
    
    return graph


In [ ]:
def test_cafe_agent():
    """
    카페 Agent 테스트 시나리오 실행
    """
    agent = build_cafe_react_agent()
    
    test_scenarios = [
        "아메리카노와 아이스 아메리카노의 차이점과 가격을 알려주세요.",
        "라떼 종류에는 어떤 메뉴들이 있고 각각의 특징은 무엇인가요?",
        "디저트 메뉴 중에서 티라미수에 대해 자세히 설명해주세요."
    ]
    
    print("=== 카페 ReAct Agent 테스트 시작 ===\n")
    
    for i, scenario in enumerate(test_scenarios, 1):
        print(f"테스트 시나리오 {i}:")
        print(f"질문: {scenario}")
        print("-" * 50)
        
        config = {"configurable": {"thread_id": f"test_thread_{i}"}}
        
        try:
            result = agent.invoke(
                {"messages": [HumanMessage(content=scenario)]},
                config=config
            )
            
            final_message = result["messages"][-1]
            if hasattr(final_message, 'content'):
                print(f"응답: {final_message.content}")
            else:
                print(f"응답: {final_message}")
            
        except Exception as e:
            print(f"오류 발생: {str(e)}")
        
        print("\n" + "="*70 + "\n")


In [ ]:
def run_interactive_cafe_agent():
    """
    대화형 카페 Agent 실행
    """
    agent = build_cafe_react_agent()
    config = {"configurable": {"thread_id": "interactive_session"}}
    
    print("카페 메뉴 상담사 Agent 시작!")
    print("질문을 입력하세요 (종료: 'quit' 또는 'exit')")
    print("-" * 50)
    
    while True:
        user_input = input("\n질문: ").strip()
        
        if user_input.lower() in ['quit', 'exit', '종료']:
            print("카페 상담을 종료합니다. 감사합니다!")
            break
        
        if not user_input:
            continue
        
        try:
            result = agent.invoke(
                {"messages": [HumanMessage(content=user_input)]},
                config=config
            )
            
            final_message = result["messages"][-1]
            print(f"\n상담사: {final_message.content}")
            
        except Exception as e:
            print(f"\n오류 발생: {str(e)}")


In [ ]:
if __name__ == "__main__":
    print("LangGraph ReAct Agent - 카페 메뉴 검색 시스템")
    print("=" * 50)
    
    if not os.getenv("GROQ_API_KEY"):
        print("GROQ_API_KEY 환경변수를 설정해주세요!")
        print("export GROQ_API_KEY='your-groq-api-key-here'")
    else:
        print("1. 테스트 시나리오 실행")
        print("2. 대화형 모드 실행")
        
        choice = input("\n선택하세요 (1 또는 2): ").strip()
        
        if choice == "1":
            test_cafe_agent()
        elif choice == "2":
            run_interactive_cafe_agent()
        else:
            print("올바른 선택지를 입력해주세요.")

LangGraph ReAct Agent - 카페 메뉴 검색 시스템
1. 테스트 시나리오 실행
2. 대화형 모드 실행
카페 메뉴 상담사 Agent 시작!
질문을 입력하세요 (종료: 'quit' 또는 'exit')
--------------------------------------------------

상담사: The price of an Americano is ₩4,500. Would you like to know more about our Americano or would you like to order one?
카페 상담을 종료합니다. 감사합니다!
